# F09 - Decision trees, random forests, and gradient boosting

**Hardware:** CPU-only · **Time:** 3 hours · **Prerequisite:** F08

Tree models split feature space into regions. A single tree is interpretable but
unstable; a forest averages many trees; gradient boosting fits trees sequentially
to reduce error. This notebook compares all three on the synthetic student table
with the same cross-validation protocol.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("pandas", pd.__version__)

pandas 2.3.3


## 1. Data and preprocessing

The tree models can handle numeric features without scaling, but the logistic
baseline needs scaling. A `ColumnTransformer` keeps preprocessing inside the
cross-validation loop, so every fold fits its own encoder and scaler.

In [2]:
def make_students(n: int = 500, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    study = rng.gamma(2.0, 2.0, n).clip(0, 20)
    attendance = rng.uniform(40, 100, n)
    prior = rng.normal(65, 12, n).clip(0, 100)
    sleep = rng.normal(7, 1.2, n).clip(3, 10)
    tutoring = rng.choice(["yes", "no"], n, p=[0.35, 0.65])
    course = rng.choice(["Math", "Physics", "CS"], n, p=[0.4, 0.3, 0.3])
    logit = -11.5 + 0.30 * study + 0.055 * attendance + 0.075 * prior + 0.20 * sleep + 1.1 * (tutoring == "yes")
    passed = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    return pd.DataFrame(
        {
            "study_hours": study.round(2),
            "attendance": attendance.round(1),
            "prior_score": prior.round(1),
            "sleep_hours": sleep.round(2),
            "tutoring": tutoring,
            "course": course,
            "passed": passed,
        }
    )


students = make_students()
X = students.drop(columns="passed")
y = students["passed"]
numeric = ["study_hours", "attendance", "prior_score", "sleep_hours"]
categorical = ["tutoring", "course"]
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ]
)
print("data shape:", X.shape, "| positive rate:", round(y.mean(), 3))
assert X.shape == (500, 6)

data shape: (500, 6) | positive rate: 0.49


## 2. A single tree and the depth trade-off

A shallow tree underfits; a deep tree can memorize the training set. The
training accuracy and cross-validated accuracy tell different stories: the gap is
the overfitting signal.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)
for depth in [2, 5, None]:
    tree = Pipeline(
        steps=[
            ("pre", preprocess),
            ("model", DecisionTreeClassifier(max_depth=depth, random_state=SEED)),
        ]
    ).fit(X_train, y_train)
    train_acc = tree.score(X_train, y_train)
    test_acc = tree.score(X_test, y_test)
    print(f"depth={depth}: train={train_acc:.3f} test={test_acc:.3f} leaves={tree.named_steps['model'].get_n_leaves()}")
assert tree.score(X_train, y_train) == 1.0

depth=2: train=0.659 test=0.656 leaves=4
depth=5: train=0.811 test=0.696 leaves=25
depth=None: train=1.000 test=0.664 leaves=86


## 3. Random forest: many trees, averaged

A random forest fits trees on bootstrap samples and random feature subsets, then
votes. It usually reduces variance compared with one deep tree. More trees
increase compute but not necessarily accuracy; `n_estimators=200` is a reasonable
default for a small table.

In [4]:
forest = Pipeline(
    steps=[
        ("pre", preprocess),
        ("model", RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=SEED, n_jobs=-1)),
    ]
).fit(X_train, y_train)
print("random forest test accuracy:", round(forest.score(X_test, y_test), 3))
assert forest.score(X_test, y_test) > 0.6

random forest test accuracy: 0.76


## 4. Gradient boosting: sequential correction

`HistGradientBoostingClassifier` fits shallow trees to the residual errors of the
current ensemble. It is often strong on tabular data, but it has more
hyperparameters and can overfit small datasets if pushed too far. Compare it with
the forest under the same folds.

In [5]:
boosting = Pipeline(
    steps=[
        ("pre", preprocess),
        ("model", HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=SEED)),
    ]
)
print("boosting test accuracy:", round(boosting.fit(X_train, y_train).score(X_test, y_test), 3))

boosting test accuracy: 0.736


## 5. Compare with cross-validation

One train/test split is noisy. Five-fold stratified cross-validation gives a
mean and spread. The table below is the evidence you should report; a single
lucky split is not a model comparison.

In [6]:
models = {
    "logistic": Pipeline([("pre", preprocess), ("model", LogisticRegression(max_iter=2000, random_state=SEED))]),
    "tree_depth_5": Pipeline([("pre", preprocess), ("model", DecisionTreeClassifier(max_depth=5, random_state=SEED))]),
    "random_forest": Pipeline([("pre", preprocess), ("model", RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=SEED, n_jobs=-1))]),
    "gradient_boosting": Pipeline([("pre", preprocess), ("model", HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=SEED))]),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
rows = []
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")
    rows.append({"model": name, "mean_accuracy": scores.mean(), "std": scores.std()})
comparison = pd.DataFrame(rows).sort_values("mean_accuracy", ascending=False)
print(comparison.round(3).to_string(index=False))
assert comparison["mean_accuracy"].max() > 0.65

            model  mean_accuracy   std
         logistic          0.742 0.027
    random_forest          0.714 0.033
gradient_boosting          0.678 0.044
     tree_depth_5          0.662 0.044


## 6. Feature importance is not causation

Impurity-based importances from a forest are useful for debugging but biased
toward high-cardinality and correlated features. Treat them as a hint, then use
permutation importance or controlled experiments on held-out data.

In [7]:
feature_names = forest.named_steps["pre"].get_feature_names_out()
importances = pd.Series(
    forest.named_steps["model"].feature_importances_, index=feature_names
).sort_values(ascending=False)
print(importances.head(8).round(3).to_string())
assert len(importances) == len(feature_names)

num__study_hours       0.267
num__attendance        0.256
num__prior_score       0.233
num__sleep_hours       0.126
cat__tutoring_yes      0.037
cat__tutoring_no       0.034
cat__course_Math       0.017
cat__course_Physics    0.016


## Check your work

- [ ] You can explain why a deep tree overfits.
- [ ] You compared models with the same folds and metric.
- [ ] You can explain what a random forest averages.
- [ ] You can explain why boosting is sequential.
- [ ] You can state why feature importance is not a causal effect.

**Next:** F10 - Metrics, overfitting, underfitting, data leakage, and cross-validation.